# EDA

In [ ]:
import os
from pathlib import Path

# If the kernel starts in .../AlmaAI/ipynb
root = Path.cwd()
if root.name == "ipynb":
    root = root.parent
os.chdir(root)

## Get Split

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

split_path = Path('cell_split/')
split_df = pd.read_csv(split_path / 'Split omnibus screen_HG - hits_concat_df.csv')
# sum(split_df['line'] == split_df['Line']) duplicate lines
split_df.drop(['line'], inplace=True, axis=1)
split_df.columns = [col.lower() for col in split_df.columns]
split_df.columns = [col.strip().replace(" ", "_") for col in split_df.columns]
split_df.shape

(310, 17)

In [2]:
# write files names to filelist.txt for download
for line in split_df['line'].tolist():
    with open('filelist.txt', 'a') as f:
        f.write(line + '\n')

### Manual List For MCFO
R9B05
R9C11
R9A07
R9B04

## Split Download
run `s3_download_fast.sh`

In [3]:
split_df.head()

,order:_reason?,unnamed:_1,class,tony_-_type,line,adult/\nlarval,main_annotator,lab,quality,sex_difference,doi_to_cite,first_author(s)_&_year,janelia_robot_id,cell_types,em_body_ids,alias,genotype
0,NaN,MB485B,mixed,various,MB485B,Adult,Yoshi Aso,Aso,2,N,10.1101/2023.09.15.557808,Shuai; 2023,2501827.0,"DL3_lPN, VM4_lvPN","1609542060, 1609883245, 1640568102\n726207450,...",GMR_48H12_BB_21-x-GMR_37E06_AV_01,w[1118]; P{y[+t7.7] w[+mC]=R48H12-p65.AD}attP4...
1,NaN,OL0022B,local/asc,? sparse,OL0022B,Adult,Aljoscha Nern,Rubin,1,N,10.7554/eLife.21022,"Wu, Nern; 2016",2135391.0,"LC10, LC10a, LC10d",NaN,GMR_35D04_XA_21-x-GMR_91B01_XD_01,w; 35D04-p65ADZp in attP40; 91B01-ZpGdbd in attP2
2,NaN,OL0027B,leg MN,?,OL0027B,Adult,Aljoscha Nern,Rubin,1,N,10.7554/eLife.21022,"Wu, Nern; 2016",2135396.0,LC13,NaN,GMR_14A11_XA_21-x-GMR_50C10_XD_01,w; 14A11-p65ADZp in attP40/CyO; 50C10-ZpGdbd i...
3,NaN,SS00313,mixed,"23B, 20A",SS00313,Adult,Aljoscha Nern,Rubin,2,N,10.1016/j.neuron.2017.03.010,Strother; 2017,2502520.0,Tm3,NaN,GMR_38C11-x-GMR_59C10,w; 38C11-p65ADZp in attP40/CyO; 59C10-ZpGdbd i...
4,NaN,SS00320,inter,various,SS00320,Adult,Aljoscha Nern,Rubin,2,N,10.7554/eLife.50901,"Davis, Nern; 2020",2502527.0,Tm4,NaN,GMR_53C02-x-GMR_60H04,w; 53C02-p65ADZp in attP40/CyO; 60H04-ZpGdbd i...


## Load cell type data from eLife
- Paper: https://elifesciences.org/articles/98405#s2
- XLSX: https://cdn.elifesciences.org/articles/98405/elife-98405-fig1-data1-v2.xlsx

In [4]:
# Load elife cell type data
elife_df = pd.read_csv(split_path / "elife-98405-fig1-data1-v2.xlsx - Cell type lines.csv")
elife_df.columns = [col.lower() for col in elife_df.columns]
elife_df.columns = [col.strip().replace(" ", "_") for col in elife_df.columns]
elife_df.head()

,line,adult/\nlarval,main_annotator,lab,quality,sex_difference,vnc_only,brain_only,doi_to_cite,first_author(s)_&_year,janelia_robot_id,cell_types,em_body_ids,alias,genotype
0,SS00043,Adult,Tanya Wolff,Rubin,1,N,No,No,10.1002/cne.24512,Wolff; 2018,2502208.0,PFNa,NaN,GMR_65B12-x-GMR_30E10,w; 65B12-p65ADZp in attP40; 30E10-ZpGdbd in attP2
1,SS00078,Adult,Tanya Wolff,Rubin,1,N,No,Yes,10.1002/cne.24512,Wolff; 2018,2502243.0,PFNd,NaN,GMR_16D01-x-GMR_15E01,w; 16D01-p65ADZp in attP40; 15E01-ZpGdbd in attP2
2,SS00090,Adult,Tanya Wolff,Rubin,1,N,No,Yes,10.1002/cne.24512,Wolff; 2018,2502255.0,EPG,NaN,GMR_19G02-x-GMR_15C03,w; 19G02-p65ADZp in attP40; 15C03-ZpGdbd in attP2
3,SS00098,Adult,Tanya Wolff,Rubin,1,N,No,No,10.1002/cne.24512,Wolff; 2018,2502263.0,EPG,NaN,GMR_19G02-x-GMR_93G12,w; 19G02-p65ADZp in attP40; 93G12-ZpGdbd in attP2
4,SS00191,Adult,Tanya Wolff,Rubin,1,N,No,Yes,10.1002/cne.24512,Wolff; 2018,2502356.0,"PFNm_a, PFNm_b, PFNp_a, PFNp_b, PFNp_c, PFNp_d...",NaN,GMR_38G07-x-GMR_37H01,w; 38G07-p65ADZp in attP40/CyO; 37H01-ZpGdbd i...


## Check to see cell types from website cell type labels vs shared spread sheet cell type labels

In [5]:
merged_df = pd.merge(elife_df, split_df, on='line', how='inner', suffixes=('_elife', '_split'))
merged_df.shape

(310, 31)

In [6]:
merged_df.head()

,line,adult/\nlarval_elife,main_annotator_elife,lab_elife,quality_elife,sex_difference_elife,vnc_only,brain_only,doi_to_cite_elife,first_author(s)_&_year_elife,...,lab_split,quality_split,sex_difference_split,doi_to_cite_split,first_author(s)_&_year_split,janelia_robot_id_split,cell_types_split,em_body_ids_split,alias_split,genotype_split
0,SS46507,Adult,Tanya Wolff,Rubin,2,Y,No,No,10.1002/cne.24512,Wolff; 2018,...,Rubin,2,Y,10.1002/cne.24512,Wolff; 2018,3027305.0,LCNp,NaN,VT040713-x-VT006486,w; VT040713-p65ADZp in attP40; VT006486-ZpGdbd...
1,SS50245,Adult,Kaiyu Wang,Dickson,2,N,No,No,10.1016/j.cub.2024.01.015,Lillvis; 2023,...,Dickson,2,N,10.1016/j.cub.2024.01.015,Lillvis; 2023,3029795.0,WN_LN_mixed,NaN,GMR_77C10-x-GMR_20A03,w[1118]; P{y[+t7.7] w[+mC]=R77C10-p65.AD}attP4...
2,SS00698,Adult,Aljoscha Nern,Rubin,2,N,No,No,10.1016/j.neuron.2013.05.024,"Tuthill, Nern; 2013",...,Rubin,1,N,10.1016/j.neuron.2013.05.024,"Tuthill, Nern; 2013",3007608.0,Lawf2,NaN,GMR_11D03-x-GMR_19C10,w[1118]; P{y[+t7.7] w[+mC]=R11D03-p65.AD}attP4...
3,SS00698,Adult,Aljoscha Nern,Rubin,2,N,No,No,10.1016/j.neuron.2013.05.024,"Tuthill, Nern; 2013",...,Rubin,1,N,10.1016/j.neuron.2013.05.024,"Tuthill, Nern; 2013",3007608.0,Lawf2,NaN,GMR_11D03-x-GMR_19C10,w[1118]; P{y[+t7.7] w[+mC]=R11D03-p65.AD}attP4...
4,SS00700,Adult,Aljoscha Nern,Rubin,2,N,No,No,10.1016/j.neuron.2013.05.024,"Tuthill, Nern; 2013",...,Rubin,1,N,10.1016/j.neuron.2013.05.024,"Tuthill, Nern; 2013",3007610.0,T1,NaN,GMR_31F10-x-GMR_65D07,w[1118]; P{y[+t7.7] w[+mC]=R31F10-p65.AD}attP4...


## Cell Type Missmatch!

In [7]:
# merged_df['cell_types_elife'] = merged_df['cell_types_elife'].where(merged_df['cell_types_elife'].notna(), '')
# merged_df['cell_types_split'] = merged_df['cell_types_split'].where(merged_df['cell_types_split'].notna(), '')
# Nan values found, replaced to be empty strings

In [8]:
merged_df[merged_df['cell_types_split'] != merged_df['cell_types_elife']].loc[:, ['line', 'cell_types_split', 'cell_types_elife']].reset_index(drop=True)

,line,cell_types_split,cell_types_elife
0,SS46507,LCNp,LCNOp
1,SS32792,NaN,NaN
2,SS35987,NaN,NaN
3,SS41518,NaN,NaN
4,SS44448,NaN,NaN
5,SS45446,NaN,NaN
6,SS51006,NaN,NaN
7,SS65693,NaN,NaN
8,SS68443,NaN,NaN
9,SS70425,NaN,NaN
